# Stochastic processes for equity and rates MC

**Start here:** This deep dive expands on `07_advanced_quant/monte_carlo_simulation.ipynb`; read the overview first for the common `TimeGrid`, `McEngine`, and pricer workflow.

This notebook surveys the **stochastic processes** behind `finstack_quant.monte_carlo`: what each parameter means, when the model is appropriate, and how Python interacts with them today.

> Python pricers in `finstack_quant.monte_carlo` currently fix the underlying dynamics to **geometric Brownian motion (GBM)** and accept risk-neutral drift parameters as numeric arguments. Processes such as Heston, Bates, Merton, CIR, and Schwartz–Smith are implemented in the Rust crate `finstack-quant-monte-carlo` and are reached from Rust or custom extensions — they are not exposed as Python classes.


## Concept

A **stochastic process** describes how state variables evolve in continuous time (often under a risk-neutral measure for pricing). In Monte Carlo you **discretize time**, draw random shocks, and simulate paths. **GBM** is the Black–Scholes workhorse; **Heston** adds stochastic volatility; **CIR** models positive short rates or variance; **jump** models add discontinuous moves; **Schwartz–Smith** is a two-factor commodity spot model; **multi-GBM** couples correlated underlyings.

## API walkthrough

High-level **GBM** pricing in Python goes through `EuropeanPricer` or `McEngine`, which assume GBM and take ``rate``, ``div_yield``, and ``vol`` directly as ``float`` arguments.

Below: run a tiny `McEngine` European call under GBM and compare against Black–Scholes. No standalone ``GbmProcess`` object is required.

In [1]:
from finstack_quant.monte_carlo import (
    EuropeanPricer,
    McEngine,
    TimeGrid,
    black_scholes_call,
    heston_satisfies_feller,
    simulate_gbm_paths,
)

gbm_params = dict(rate=0.05, div_yield=0.0, vol=0.20)
heston_params = dict(
    rate=0.05,
    div_yield=0.0,
    v0=0.04,
    kappa=2.0,
    theta=0.04,
    xi=0.3,
    rho=-0.7,
)
feller_ok = heston_satisfies_feller(heston_params["kappa"], heston_params["theta"], heston_params["xi"])
print(f"GBM params: {gbm_params}")
print(f"Heston params: {heston_params}")
print(f"Heston Feller (2*kappa*theta > xi^2): {feller_ok}")

grid = TimeGrid(t_max=1.0, num_steps=252)
engine = McEngine(num_paths=25_000, time_grid=grid, seed=42)
mc_call = engine.price_european_call(100.0, 100.0, **gbm_params)
bs = black_scholes_call(100.0, 100.0, 0.05, 0.0, 0.20, 1.0)
print(f"McEngine ATM call mean: {mc_call.mean.amount:.6f} (stderr={mc_call.stderr:.6f})")
print(f"Black–Scholes call:       {bs:.6f}")

pr = EuropeanPricer(num_paths=25_000, seed=42)
eu = pr.price_call(100.0, 100.0, 0.05, 0.0, 0.20, 1.0, num_steps=252)
print(f"EuropeanPricer ATM call:  {eu.mean.amount:.6f}")

path_summary = simulate_gbm_paths(
    spot=100.0,
    expiry=1.0,
    num_steps=12,
    num_paths=3,
    seed=42,
    **gbm_params,
)
print("Captured GBM path shape:", len(path_summary.paths), "x", len(path_summary.times))
print("Terminal spots:", [round(path[-1], 4) for path in path_summary.paths])

GBM params: {'rate': 0.05, 'div_yield': 0.0, 'vol': 0.2}
Heston params: {'rate': 0.05, 'div_yield': 0.0, 'v0': 0.04, 'kappa': 2.0, 'theta': 0.04, 'xi': 0.3, 'rho': -0.7}
Heston Feller (2*kappa*theta > xi^2): True
McEngine ATM call mean: 10.502130 (stderr=0.092865)
Black–Scholes call:       10.450584


EuropeanPricer ATM call:  10.502130
Captured GBM path shape: 3 x 13
Terminal spots: [109.8744, 90.279, 114.8215]


## Parameter cheat-sheet

A short reference for the processes implemented inside the Rust crate (not surfaced as Python classes today):

- **GBM** — log-normal spot; Black–Scholes workhorse when vol is constant.
- **Heston** — stochastic variance with spot-vol correlation; captures equity smiles.
- **Brownian (arithmetic)** — local-vol / rate building block; can go negative.
- **CIR** — mean-reverting positive process; short rates or Heston variance.
- **Merton / Bates** — Merton adds log-normal jumps; Bates = Heston + jumps for short-dated skew.
- **Schwartz–Smith** — two-factor log-spot decomposition; commodity forward curves.
- **Multi-GBM** — correlated diffusions for baskets and spread options.

Use `heston_satisfies_feller` so parameter validation and the strict Heston Feller check remain Rust-owned.

In [2]:
from finstack_quant.monte_carlo import heston_satisfies_feller

heston = dict(kappa=2.0, theta=0.04, xi=0.3)
print(
    "Heston Feller:",
    heston_satisfies_feller(heston["kappa"], heston["theta"], heston["xi"]),
)

Heston Feller: True


## Heston stochastic-volatility pricing

`price_heston_call` / `price_heston_put` simulate the full Heston model (mean-reverting variance with `kappa`, `theta`, `vol_of_vol`, spot/vol correlation `rho`, and initial variance `v0`) and return a `MoneyEstimate`.

In [3]:
from finstack_quant.monte_carlo import price_heston_call, price_heston_put

hc = price_heston_call(
    spot=100.0,
    strike=100.0,
    rate=0.05,
    div_yield=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.30,
    rho=-0.70,
    v0=0.04,
    expiry=1.0,
    num_paths=20_000,
    seed=42,
)
hp = price_heston_put(
    spot=100.0,
    strike=100.0,
    rate=0.05,
    div_yield=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.30,
    rho=-0.70,
    v0=0.04,
    expiry=1.0,
    num_paths=20_000,
    seed=42,
)
print(f"Heston call: {hc.mean} (stderr={hc.stderr:.4f})")
print(f"Heston put:  {hp.mean} (stderr={hp.stderr:.4f})")

Heston call: USD 10.42 (stderr=0.0868)
Heston put:  USD 5.44 (stderr=0.0697)


## Takeaways

- **GBM** is the default workhorse for equity-style Monte Carlo in this stack; `McEngine` prices Europeans under GBM with an **exact** time stepper.
- Process parameters are plain floats in the Python API — no wrapper objects to carry state between construction and pricing.
- **Heston** and **CIR** parameters should still satisfy their respective **Feller** conditions to keep variance strictly positive; the check is a one-liner.
- **Jumps** (Merton, Bates) matter when short-tenor returns show **fat tails**; **Bates** combines stochastic vol and jumps. These are available in the Rust crate.
- **Schwartz–Smith** and **multi-GBM** are the natural starting points for **commodity** and **correlation** structures, respectively, again via Rust.